# YOLO nodule detector: training on Google Colab

Jairus

Trains the detector with `pipeline/6train_yolo.py` on a free Colab GPU. Every epoch is saved to `My Drive/LIDC/yolo_runs/`, so a disconnect loses at most one epoch.

**Rule: never train the same run in two Colab tabs at once.** Both would write to the same run and overwrite each other. As a safety net, Resume refuses to start if the run was updated in the last 20 minutes.

**Before you start**
1. In My Drive, the folder **`LIDC`** must hold `yolo_lidc_train_jpg.zip`, the latest `lidc_code.zip` and `dataset.yaml` (both from the `colab_upload` folder in the project on the Mac; replace any older copies).
2. In Colab: **Runtime → Change runtime type → T4 GPU**.
3. Run cells 1–4, then **Train** for a brand-new run or **Resume** to carry on an existing one.

**If Colab disconnects** (free Colab stops after ~90 min of inactivity or ~12 h in total): reconnect, run cells 1–4 again, then **Resume**.

**Main model vs control**: train `yolo26s.pt` first, then change `MODEL` to `yolo11s.pt` and train again. Leave every other setting the same so the comparison is fair.

## 1. Settings

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/LIDC"   # folder in Google Drive holding the two zips and yolo_runs/
DATA_ZIP = "yolo_lidc_train_jpg.zip"        # or "yolo_lidc_train.zip" for the PNG version
MODEL = "yolo26s.pt"                         # main model; "yolo11s.pt" for the control
EPOCHS = 100                                 # early stopping usually ends it sooner
IMGSZ = 640
BATCH = 16                                   # keep the same for both models
FORCE = False                                # True only if you're sure no other session is training this run

## 2. GPU and Google Drive

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

## 3. Install

In [ ]:
!pip install -q "ultralytics>=8.4" pydicom

## 4. Unpack the data and code onto Colab's own disk

Training reads ~22,000 images every epoch; reading them straight from Google Drive would be very slow, so they're copied to the Colab machine first (takes a few minutes, only once per session). `dataset.yaml` from the `LIDC` folder is then copied into the unzipped `yolo_lidc` folder, where the YOLO library expects it.

In [ ]:
import os, shutil, time, zipfile
from pathlib import Path

for name, target, check in [
        ("lidc_code.zip", "/content", "/content/code/pipeline/ct.py"),
        (DATA_ZIP, "/content/processed", "/content/processed/yolo_lidc/val_monitor.txt")]:
    if Path(check).exists():
        print(f"{name}: already unpacked")
        continue
    t0 = time.time()
    local = f"/content/{name}"
    shutil.copy(f"{DRIVE_DIR}/{name}", local)
    with zipfile.ZipFile(local) as z:
        z.extractall(target)
    os.remove(local)
    print(f"{name}: unpacked in {time.time() - t0:.0f}s")

# dataset.yaml lives next to the zip in Drive, not inside it: put it in the unzipped folder
yaml_src, yaml_dst = Path(f"{DRIVE_DIR}/dataset.yaml"), Path("/content/processed/yolo_lidc/dataset.yaml")
if yaml_src.exists():
    shutil.copy(yaml_src, yaml_dst)
    print(f"dataset.yaml: copied to {yaml_dst}")
else:
    print(f"dataset.yaml: not found in {DRIVE_DIR} (only needed to use the YOLO library directly)")

os.environ["LIDC_PROCESSED"] = "/content/processed"          # where the scripts look for the dataset
os.environ["LIDC_YOLO_RUNS"] = f"{DRIVE_DIR}/yolo_runs"      # runs go to Drive, so they survive
os.chdir("/content/code/pipeline")
WORKERS = os.cpu_count()

n_train = len(list(Path("/content/processed/yolo_lidc/images/train").glob("*.*")))
print(f"{n_train} training images (should be 22050), runs saved to {os.environ['LIDC_YOLO_RUNS']}")

import pandas as pd
RUN_NAME = f"{Path(MODEL).stem}_img{IMGSZ}_e{EPOCHS}_b{BATCH}"

def epochs_done(run):
    csv = run / "results.csv"
    return len(pd.read_csv(csv)) if csv.exists() else 0

def runs_of_model():
    return list(Path(os.environ["LIDC_YOLO_RUNS"]).glob(RUN_NAME + "*"))

def latest_run():
    """The run of MODEL with the most epochs."""
    runs = runs_of_model()
    assert runs, f"no run of {RUN_NAME} in {os.environ['LIDC_YOLO_RUNS']}: use the Train cell"
    return max(runs, key=epochs_done)

def check_not_running_elsewhere(run):
    age_min = (time.time() - (run / "weights/last.pt").stat().st_mtime) / 60
    print(f"{run.name}: {epochs_done(run)} epochs, last.pt updated {age_min:.0f} min ago")
    if age_min < 20 and not FORCE:
        raise SystemExit("Updated in the last 20 minutes: is another Colab tab still training it? "
                         "Stop that first, or set FORCE = True.")


## 5. (Optional) Timing test

2 short epochs on 5% of the training images. Prints how long one full epoch takes on this GPU.

In [ ]:
!python 6train_yolo.py --smoke --model {MODEL} --imgsz {IMGSZ} --device 0 --batch {BATCH} --workers {WORKERS}

## 6. Train (brand-new run only)

Starts a new run of `MODEL`. If one already exists, it refuses: use **Resume** instead.

In [ ]:
if runs_of_model():
    print(f"A run of {RUN_NAME} already exists ({latest_run().name}, {epochs_done(latest_run())} epochs): use the Resume cell.")
else:
    !python 6train_yolo.py --model {MODEL} --epochs {EPOCHS} --imgsz {IMGSZ} --device 0 --batch {BATCH} --workers {WORKERS}

## Resume

Carries on from the latest saved epoch of `MODEL`.

In [ ]:
run = latest_run()
check_not_running_elsewhere(run)
print("resuming", run)
!python 6train_yolo.py --resume "{run}" --device 0 --workers {WORKERS}

## 7. Results

YOLO's own per-epoch check on `val_monitor.txt`. This is only a progress check; the numbers for the report come from `7eval_yolo.py` (per nodule, every slice).

In [ ]:
run = latest_run()
r = pd.read_csv(run / "results.csv")
r.columns = r.columns.str.strip()
display(r[["epoch", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(10))
print("best epoch:", int(r.loc[r["metrics/mAP50-95(B)"].idxmax(), "epoch"]))
print("best weights:", run / "weights" / "best.pt")

## Next

1. Train the control: set `MODEL = "yolo11s.pt"` in cell 1, run cell 1, then cell 6.
2. Download each run folder from `My Drive/LIDC/yolo_runs/` to the Mac (e.g. `~/Downloads/yolo_runs/`), not to the Expansion drive.
3. On the Mac, from `pipeline/`: `python 7eval_yolo.py --weights ~/Downloads/yolo_runs/<run>/weights/best.pt --split val`
   (reads the val images from the drive, writes its results next to the weights on the Mac)